In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q12/q12_rewrite/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

### cell 2 (optimized for cudf) ###

# filter bounds (kept on GPU)
date1 = pd.Timestamp("1994-01-01")
date2 = pd.Timestamp("1995-01-01")

# 1) filter lineitem and project only the columns needed for the join and grouping
flineitem = (
    lineitem
    [(lineitem.L_RECEIPTDATE >= date1)
     & (lineitem.L_RECEIPTDATE < date2)
     & (lineitem.L_COMMITDATE   < date2)
     & (lineitem.L_SHIPDATE      < date2)
     & (lineitem.L_SHIPDATE      < lineitem.L_COMMITDATE)
     & (lineitem.L_COMMITDATE    < lineitem.L_RECEIPTDATE)
     & lineitem.L_SHIPMODE.isin(["MAIL", "SHIP"]) ]
    [["L_ORDERKEY", "L_SHIPMODE"]]
)

# 2) project only the columns we need from orders
small_orders = orders[["O_ORDERKEY", "O_ORDERPRIORITY"]]

# 3) join on the minimal set of columns
jn = flineitem.merge(
    small_orders,
    left_on="L_ORDERKEY",
    right_on="O_ORDERKEY"
)

# 4) build the urgency flags (all on GPU)
jn["is_urgent"]     = jn.O_ORDERPRIORITY.isin(["1-URGENT", "2-HIGH"])
jn["is_not_urgent"] = ~jn["is_urgent"]

# 5) group and aggregate in one pass
total = jn.groupby("L_SHIPMODE", as_index=False).agg(
    g1=("is_urgent",     "sum"),
    g2=("is_not_urgent", "sum")
)

print(total)

  L_SHIPMODE    g1    g2
0       MAIL  6202  9324
1       SHIP  6200  9262
CPU times: user 233 ms, sys: 128 ms, total: 362 ms
Wall time: 368 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q12/rewritten/o4_mini_high_small_q12/checkpoints/post_cell_2_try_1.pickle